# Gold Fact: Holdings (fact_holdings)

- **Purpose**: Transforms `silver.holdings` into the final `gold.fact_holdings` table to track account position snapshots (quantities) after every trade.
- **Business Context**: PWG Pipeline - Trade Domain. This table utilizes a **cascading keys pattern**. Instead of performing expensive and risky point-in-time temporal joins, it inherits all dimension surrogate keys directly from the central `dim_trade` table via a single join on `HH_T_ID` (Current Trade). 
- **Execution Frequency**: Per Batch (Incremental)
- **Inputs**: `silver.holdings`, `gold.dim_trade`
- **Outputs**: `gold.fact_holdings`
- **Dependencies**: **CRITICAL - MUST RUN AFTER dim_trade IS 100% COMPLETE.** Never run in parallel with `dim_trade`.
- **Expected Row Count**: 1,206,578 (Validated via HH_RECORDS B1+B2+B3)

> Imported our operaitons notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
# configuring widgets for ease of use and reusability
dbutils.widgets.text("env_catalog", "charles_schwab_retailbrokerage_dev_team_lemma")
catalog = dbutils.widgets.get("env_catalog")

# Source tables 
silver_tbl      = f"{catalog}.silver.holdings"
dim_trade_tbl   = f"{catalog}.gold.dim_trade"

# Target tables
gold_tbl         = f"{catalog}.gold.fact_holdings"

In [0]:
# Importing the required functions 
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Logging functions for initial load
l_df = spark.sql(f"SELECT * FROM {silver_tbl} ORDER BY _load_ts DESC LIMIT 1")
carried_batch = spark.sql(f"SELECT _batch_id FROM {silver_tbl} ORDER BY _load_ts DESC LIMIT 1").first()[0]
carried_run_id = str(l_df.select("_run_id").first()[0])

log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_fact_holdings', 'Starting processing for standalone gold fact_holdings')

start_pipeline_run(spark, carried_run_id, carried_batch)

log_domain_run_status(spark, carried_run_id, carried_batch, 'TRADE', 'RUNNING')

In [0]:
# Reading data into dataframes
df_holdings = spark.table(silver_tbl).alias("h")
dim_trade = spark.table(dim_trade_tbl).alias("t")


# Storing source count from here 
source_count = df_holdings.count()

In [0]:
# Simple join to dim_trade to get eveything holdings need to make fact table form dim_trade
df_joined = df_holdings.join(dim_trade, col("h.HH_T_ID") == col("t.TradeID"), "left")

In [0]:
df_joined.printSchema()

In [0]:
df_mapped = df_joined.select(
    col("h.HH_H_T_ID").alias("TradeID"),
    col("h.HH_T_ID").alias("CurrentTradeID"),
    col("t.SK_CustomerID"),
    col("t.SK_AccountID"),
    col("t.SK_SecurityID"),
    col("t.SK_CompanyID"),
    col("t.SK_CreateDateID").alias("SK_DateID"),
    col("t.SK_CreateTimeID").alias("SK_TimeID").cast("string"),
    col("t.TradePrice").alias("CurrentPrice"),
    col("h.HH_AFTER_QTY").alias("CurrentHolding"),
    col("t._batch_id"),
    current_timestamp().alias("_load_ts")
)

In [0]:
df_mapped.createOrReplaceTempView("raw_holdings")

df_mapped.limit(0).write.format("delta").mode("ignore").saveAsTable(gold_tbl)

ip_merge = spark.sql(f"""
          merge into {gold_tbl} as t
          using raw_holdings as s
          on t.TradeID = s.TradeID and s.CurrentTradeID = t.CurrentTradeID
          when matched then update set *
          when not matched then insert *
          """)
display(ip_merge)

In [0]:
target_count = spark.table(gold_tbl).count()
null_count = spark.sql(f"SELECT COUNT(*) FROM {gold_tbl} WHERE SK_AccountID IS NULL").first()[0]

log_dq_result(spark, carried_run_id, gold_tbl, "Null SK_AccountID Check", null_count, source_count)
log_domain_run_status(spark, carried_run_id, carried_batch, 'TRADE', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_fact_holdings', 'Successfully completed standalone gold fact_holdings')
log_gold_recon(spark, carried_run_id, gold_tbl, expected_count=1302248, actual_count=target_count)

In [0]:
try:
    # Operations Logging

    # Extract the carry-forwarded _run_id from dataframe
    carried_run_id = str(df_joined.select("h._run_id").first()[0])

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id="ALL",
        domain="TRADE",
        table_name="fact_holdings",
        source_layer="silver",
        target_layer="gold",
        source_count=int(source_count),
        target_count=int(target_count)
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch="ALL",
        layer="gold",
        table_name="fact_holdings",
        operation="OVERWRITE",
        rows_affected=int(target_count)
    )

    print("Done GOLD")
except Exception as e:
    print(f"Error during operations logging: {e}")